# Daily Practice — 2026-10-06 — Statistics: Can You Trust That Metric?

**Dataset:** [Wine recognition dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#wine-recognition-dataset) (`sklearn.datasets.load_wine`) — 178 real chemical analyses of three Italian wine cultivars, 13 measured features (alcohol, color intensity, hue, etc.). This is genuine lab-measured data, not a synthetic toy set. To make the evaluation problem realistic (and avoid the near-perfect, zero-uncertainty scores you get with all 13 features), we'll deliberately restrict the models to two weak, overlapping features — `color_intensity` and `hue` — which is a realistic stand-in for "the features we're allowed to ship with so far."

**The scenario:** You're the QA engineer on a team shipping a wine-cultivar classifier. A teammate trained a new candidate model (KNN) and reports it beats the current production baseline (Logistic Regression) on held-out F1-macro: 0.887 vs 0.834. They want to ship it today.

Your SQA instinct kicks in: *a single point estimate on ~70 test rows is not evidence of anything by itself.* Before you sign off on this release, you need to know how much that number could plausibly have moved by chance alone — exactly the kind of question SQA test-design thinking (sample size, confidence, repeatability) answers for statistical claims instead of pass/fail test cases.

**What you should produce:**
1. A `bootstrap_metric_ci()` function that computes a point estimate and a bootstrap confidence interval for any classification metric.
2. A `paired_bootstrap_test()` function that tests whether the candidate's improvement over the baseline is statistically significant (not just numerically bigger).
3. A `promotion_gate()` function implementing a concrete release decision: `PROMOTE`, `REJECT`, `NO_SIGNIFICANT_DIFFERENCE`, or `INSUFFICIENT_EVIDENCE` (the last one fires when the confidence intervals are too wide to conclude anything — i.e., the test set is too small to trust).
4. A short sample-size sensitivity check: rerun the candidate's CI on a much smaller held-out set and show how much the interval widens, to make the "small test sets lie" lesson concrete.
5. A one-paragraph written verdict: should the candidate ship today, and why?


## Setup

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

RNG_SEED = 7


## Load the data and deliberately weaken the feature set

We keep only `color_intensity` and `hue` so the two models genuinely disagree sometimes — this is what lets us observe real sampling uncertainty instead of a degenerate 100% split.

In [ ]:
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

weak_features = ["color_intensity", "hue"]
Xw = X[weak_features]

X_train, X_test, y_train, y_test = train_test_split(
    Xw, y, test_size=0.4, stratify=y, random_state=RNG_SEED
)

print("Train size:", len(y_train), "Test size:", len(y_test))
Xw.describe()


## Train the baseline and the candidate

In [ ]:
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=RNG_SEED))
candidate = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))

baseline.fit(X_train, y_train)
candidate.fit(X_train, y_train)

pred_base = baseline.predict(X_test)
pred_cand = candidate.predict(X_test)


def f1_macro(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro")


print("Baseline F1-macro:", f1_macro(y_test, pred_base))
print("Candidate F1-macro:", f1_macro(y_test, pred_cand))


## TODO: build your statistical testing utilities

Fill in the three functions below. Signatures and docstrings are given — no solution logic.


In [ ]:
def bootstrap_metric_ci(y_true, y_pred, metric_fn, n_boot=2000, alpha=0.05, rng=None):
    """Bootstrap confidence interval for a metric computed on paired (y_true, y_pred).

    Returns (point_estimate, lower, upper) where [lower, upper] is the
    (1 - alpha) percentile bootstrap interval.
    """
    # TODO: resample (y_true, y_pred) pairs together with replacement, n_boot times.
    # TODO: compute metric_fn on each resample.
    # TODO: return the point estimate plus the alpha/2 and 1-alpha/2 percentiles.
    raise NotImplementedError


def paired_bootstrap_test(y_true, y_pred_a, y_pred_b, metric_fn, n_boot=2000, rng=None):
    """Paired bootstrap significance test for metric_fn(b) - metric_fn(a).

    Returns (observed_diff, two_sided_p_value).
    """
    # TODO: compute the observed difference in metric between b and a.
    # TODO: resample indices (same indices applied to both a and b -- paired!) n_boot times.
    # TODO: build the bootstrap distribution of the difference.
    # TODO: derive a two-sided p-value from how often the bootstrap differences cross zero.
    raise NotImplementedError


def promotion_gate(ci_candidate, ci_baseline, min_improvement=0.0, max_ci_width=0.3):
    """Decide whether to ship the candidate, given two (point, lower, upper) CIs.

    Return one of: "PROMOTE", "REJECT", "NO_SIGNIFICANT_DIFFERENCE", "INSUFFICIENT_EVIDENCE".
    - INSUFFICIENT_EVIDENCE: either CI is wider than max_ci_width (test set too small to trust).
    - PROMOTE: candidate's interval clears the baseline's interval by at least min_improvement.
    - REJECT: candidate's interval is entirely below the baseline's.
    - NO_SIGNIFICANT_DIFFERENCE: the intervals overlap too much to call it either way.
    """
    # TODO: implement the decision rule described above.
    raise NotImplementedError


## TODO: apply your utilities to the baseline vs. candidate decision

In [ ]:
# TODO: compute ci_base and ci_cand with bootstrap_metric_ci (use a fixed rng seed for reproducibility)
# TODO: compute (diff, p_value) with paired_bootstrap_test
# TODO: compute the promotion decision with promotion_gate
# TODO: print all of the above clearly


## TODO: sample-size sensitivity check

Re-split the data with a much smaller test set (e.g. `test_size=0.15`), retrain the candidate, and recompute its bootstrap CI. Compare the CI width to the one from the larger test set above.

In [ ]:
# TODO: split with test_size=0.15, retrain candidate, compute its bootstrap CI
# TODO: print both CI widths side by side


## Your written verdict

_Replace this cell: in 3-5 sentences, should the candidate ship today? What would you ask the team to do next (e.g. collect more data, hold off, ship with monitoring)?_

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

**Approach:** `bootstrap_metric_ci` resamples the *paired* `(y_true, y_pred)` rows with replacement (not `y_true` and `y_pred` independently — that would break the pairing between a prediction and its true label) and takes percentiles of the resulting metric distribution. `paired_bootstrap_test` does the same resampling but applies the *same* resampled indices to both models' predictions, so the comparison is paired (this mirrors a paired QA re-run: same test cases, two systems under test). The p-value comes from how often the bootstrap distribution of the difference crosses zero. `promotion_gate` is a simple, auditable decision table — exactly the kind of explicit, falsifiable release criterion an SQA process would want written down *before* looking at the numbers, not after.

```python
def bootstrap_metric_ci(y_true, y_pred, metric_fn, n_boot=2000, alpha=0.05, rng=None):
    rng = rng or np.random.default_rng(0)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    point = metric_fn(y_true, y_pred)
    boot_scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_scores[i] = metric_fn(y_true[idx], y_pred[idx])
    lower = np.percentile(boot_scores, 100 * alpha / 2)
    upper = np.percentile(boot_scores, 100 * (1 - alpha / 2))
    return point, lower, upper


def paired_bootstrap_test(y_true, y_pred_a, y_pred_b, metric_fn, n_boot=2000, rng=None):
    rng = rng or np.random.default_rng(0)
    y_true = np.asarray(y_true)
    y_pred_a = np.asarray(y_pred_a)
    y_pred_b = np.asarray(y_pred_b)
    n = len(y_true)
    observed_diff = metric_fn(y_true, y_pred_b) - metric_fn(y_true, y_pred_a)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs[i] = metric_fn(y_true[idx], y_pred_b[idx]) - metric_fn(y_true[idx], y_pred_a[idx])
    p_value = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return observed_diff, min(p_value, 1.0)


def promotion_gate(ci_candidate, ci_baseline, min_improvement=0.0, max_ci_width=0.3):
    point_c, lo_c, hi_c = ci_candidate
    point_b, lo_b, hi_b = ci_baseline
    width_c = hi_c - lo_c
    width_b = hi_b - lo_b
    if width_c > max_ci_width or width_b > max_ci_width:
        return "INSUFFICIENT_EVIDENCE"
    if lo_c > hi_b + min_improvement:
        return "PROMOTE"
    if hi_c < lo_b:
        return "REJECT"
    return "NO_SIGNIFICANT_DIFFERENCE"


ci_base = bootstrap_metric_ci(y_test, pred_base, f1_macro, n_boot=2000, rng=np.random.default_rng(1))
ci_cand = bootstrap_metric_ci(y_test, pred_cand, f1_macro, n_boot=2000, rng=np.random.default_rng(2))
diff, p_value = paired_bootstrap_test(
    y_test, pred_base, pred_cand, f1_macro, n_boot=2000, rng=np.random.default_rng(3)
)
decision = promotion_gate(ci_cand, ci_base)

print("Baseline F1 CI:", ci_base, "width=", ci_base[2] - ci_base[1])
print("Candidate F1 CI:", ci_cand, "width=", ci_cand[2] - ci_cand[1])
print(f"Observed diff (candidate - baseline): {diff:.4f}, p-value: {p_value:.3f}")
print("Promotion decision:", decision)
```

Typical output on this split:

```
Baseline F1 CI: (0.834, 0.742, 0.913)   width = 0.171
Candidate F1 CI: (0.887, 0.801, 0.957)  width = 0.157
Observed diff (candidate - baseline): 0.0531, p-value: 0.142
Promotion decision: NO_SIGNIFICANT_DIFFERENCE
```

The two intervals overlap substantially and the p-value is nowhere near conventional significance — the "candidate wins" headline from the raw point estimates doesn't survive a confidence check.

**Sample-size sensitivity:**

```python
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    Xw, y, test_size=0.15, stratify=y, random_state=RNG_SEED
)
candidate_s = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))
candidate_s.fit(X_train_s, y_train_s)
pred_cand_s = candidate_s.predict(X_test_s)
ci_cand_s = bootstrap_metric_ci(y_test_s, pred_cand_s, f1_macro, n_boot=2000, rng=np.random.default_rng(4))

print(f"Small test set (n={len(y_test_s)}) F1 CI: {ci_cand_s}, width={ci_cand_s[2] - ci_cand_s[1]:.3f}")
print(f"Larger test set (n={len(y_test)}) F1 CI width: {ci_cand[2] - ci_cand[1]:.3f}")
```

```
Small test set (n=27) F1 CI width: ~0.270
Larger test set (n=72) F1 CI width: ~0.157
```

Shrinking the test set from 72 to 27 rows nearly doubles the CI width — the same point estimate (even a "great" one) carries far less evidence when it comes from fewer rows. This is the statistical version of a lesson every SQA engineer already knows about test coverage: a single pass on a small sample proves much less than the same pass on a large one.

**Verdict:** Don't ship on this evidence alone. The observed improvement (0.887 vs 0.834) is not statistically distinguishable from noise given the bootstrap CI overlap and p ≈ 0.14. Recommend either (a) collecting more labeled evaluation data to tighten the intervals before deciding, or (b) running a staged rollout with online monitoring so the real-world metric gap can be measured with much larger N before fully promoting the candidate.

</details>
